In [ ]:
from pathlib import Path

from helpers._helpers import run_experiment_ablation, run_sweeps_for_process_list
import numpy as np

from olbootstrap.online_bootstrap._block_bootstrap import (
    BlockBootstrap,
)
from olbootstrap.online_bootstrap._online_ar_bootstrap import OnlineARBootstrap
from olbootstrap.online_bootstrap._online_gaussian_bootstrap import (
    OnlineGaussianMixtureAsympCSSmoothedBootstrap,
)
from olbootstrap.study._study import (
    UniformCoverageStudyWithTesting,
)
from olbootstrap.synthetic_time_series._ar1 import AR1Process

In [ ]:
seed_master = 1234
rng = np.random.default_rng(seed_master)

base_proc = AR1Process(
    mean=0.0,
    phi=0.3,
    noise_std=1.0,
    seasonal_amplitude=0.4,
    seasonal_period=400,
    seasonal_phase=0.0,
    rng=rng,
)

base_exp_kwargs = {
    "B": 400,
    "record_every": 1,
    "smoothing_method": "ewma",
    "method_label": "ARmmult",
    "bootstrap_cls": OnlineARBootstrap,
    "use_variance_smoothing": False,
}

res_dict = UniformCoverageStudyWithTesting.run_power_sweep(
    base_process_template=base_proc,
    etas=[2 / 20, 2 / 50, 2 / 100],
    phis=[0.3, 0.6],
    trend_slopes=[0.0001, 0.0005, 0.001],
    sample_size=3500,
    n_series=150,
    burn_in=500,
    var_warmup=400,
    alpha=0.1,
    base_exp_kwargs=base_exp_kwargs,
    outdir=Path("..") / "experiments" / "power_ewma_alpha-0p1",
    seed=seed_master,
    parallel=True,
    n_jobs=-1,
    transform="student",
    transform_power=1 / 3,
    rho_power=-1 / 3,
    save=True,
)
print("saved", len(res_dict), "runs")

In [ ]:
exp_kwargs_overrides = [
    {
        "bootstrap_cls": OnlineARBootstrap,
        "method_label": "ARmmult",
        "transform": "student",
    },
    {
        "bootstrap_cls": OnlineGaussianMixtureAsympCSSmoothedBootstrap,
        "method_label": "GaussMix",
        "transform": "student",
    },
    {
        "bootstrap_cls": OnlineARBootstrap,
        "method_label": "ARmrho0",
        "rho_power": 0.0,
        "transform": "gauss",
    },
    {
        "bootstrap_cls": BlockBootstrap,
        "method_label": "Block",
        "transform": "student",
    },
]

In [ ]:
outdir = Path("..") / "experiments"
outdir.mkdir(parents=True, exist_ok=True)

seed_master = 1234
phis = [0.3, 0.6]

sample_size = 3500
smoothing_grid = [2 / 10, 2 / 20, 2 / 50, 2 / 100, 2 / 250]
n_series = 150
burn_in = 500
var_warmup = 400

# panel_names = ["Trend + seasonality"]
panel_names = ["Stationary", "Trend + seasonality", "Trend + shocks"]


def make_panel_process(panel_name: str, phi: float, rng: np.random.Generator):
    if panel_name == "Stationary":
        return AR1Process(
            mean=0.0,
            phi=phi,
            noise_std=1.0,
            trend_slope=0.0,
            seasonal_amplitude=0.0,
            seasonal_period=None,
            seasonal_phase=0.0,
            rng=rng,
        )
    if panel_name == "Trend + seasonality":
        return AR1Process(
            mean=0.0,
            phi=phi,
            noise_std=1.0,
            trend_slope=0.001,
            seasonal_amplitude=0.4,
            seasonal_period=400,
            seasonal_phase=0.0,
            rng=rng,
        )
    if panel_name == "Trend + shocks":
        return AR1Process(
            mean=0.0,
            phi=phi,
            noise_std=1.0,
            trend_slope=0.001,
            seasonal_amplitude=0.0,
            seasonal_period=None,
            seasonal_phase=0.0,
            rng=rng,
            shock_type="permanent",
            jump_prob=0.005,
            jump_scale=2.0,
        )
    raise ValueError(f"Unknown panel: {panel_name}")


setups = [
    # ("ewma", 0.1),
    # ("ewma", 0.2),
    # ("brown", 0.1),
    ("brown", 0.2),
]

for setup_idx, (smooth_method, alpha) in enumerate(setups):
    base_exp_kwargs = {
        "B": 400,
        "record_every": 1,
        "smoothing_method": smooth_method,
    }

    setup_dir = outdir / f"smooth-{smooth_method}_alpha-{str(alpha).replace('.', 'p')}"
    setup_dir.mkdir(parents=True, exist_ok=True)

    for panel_idx, panel_name in enumerate(panel_names):
        processes = []
        labels = []
        for phi_idx, phi in enumerate(phis):
            rng = np.random.default_rng(
                seed_master + 10_000 * setup_idx + 1_000 * panel_idx + 10 * phi_idx
            )
            proc = make_panel_process(panel_name, phi, rng)
            processes.append(proc)
            labels.append(rf"phi={phi:g}")

        _ = run_sweeps_for_process_list(
            processes,
            labels=labels,
            outdir=setup_dir / panel_name,
            seed=seed_master + 100 * setup_idx + panel_idx,
            sample_size=sample_size,
            smoothing_grid=smoothing_grid,
            n_series=n_series,
            burn_in=burn_in,
            alpha=alpha,
            var_warmup=var_warmup,
            base_exp_kwargs=base_exp_kwargs,
            exp_kwargs_overrides=exp_kwargs_overrides,
            progress=True,
            parallel=True,
            n_jobs=-1,
        )

print("Done. Results saved under:", outdir.resolve())

In [ ]:
phi = 0.5
ar1_base = AR1Process(
    mean=0.0,
    phi=phi,
    noise_std=1.0,
    trend_slope=0.0,
    seasonal_amplitude=0.0,
    seasonal_period=None,
    seasonal_phase=0.0,
    rng=np.random.default_rng(0),
)

sample_size = 3500
smoothing_grid = [2 / 10, 2 / 20, 2 / 50, 2 / 100, 2 / 250]
n_series = 150
burn_in = 500
alpha = 0.1
var_warmup_fixed = 400
seed_master = 1234

base_exp_kwargs = {
    "B": 200,
    "record_every": 1,
    "smoothing_method": "ewma",
}

In [ ]:
B_values = [20, 50, 100, 200]
res_B = run_experiment_ablation(
    base_process_template=ar1_base,
    param_name="B",
    param_values=B_values,
    outdir=Path("..") / "experiments" / "AR1_phi0p5_ablate_B",
    sample_size=sample_size,
    smoothing_grid=smoothing_grid,
    n_series=n_series,
    burn_in=burn_in,
    alpha=alpha,
    var_warmup=var_warmup_fixed,
    base_exp_kwargs=base_exp_kwargs,
    transform="student",
    transform_power=1.0 / 3.0,
    rho_power=-1.0 / 3.0,
    seed_master=seed_master,
)

In [ ]:
vw_values = [20, 100, 200, 400]
res_vw = run_experiment_ablation(
    base_process_template=ar1_base,
    param_name="var_warmup",
    param_values=vw_values,
    outdir=Path("..") / "experiments" / "AR1_phi0p5_ablate_vw",
    sample_size=sample_size,
    smoothing_grid=smoothing_grid,
    n_series=n_series,
    burn_in=burn_in,
    alpha=alpha,
    var_warmup=var_warmup_fixed,
    base_exp_kwargs=base_exp_kwargs,
    transform="student",
    transform_power=1.0 / 3.0,
    rho_power=-1.0 / 3.0,
    seed_master=seed_master,
)

In [ ]:
tp_values = [1 / 4, 1 / 3, 1 / 2, 1.0]
res_tp = run_experiment_ablation(
    base_process_template=ar1_base,
    param_name="transform_power",
    param_values=tp_values,
    outdir=Path("..") / "experiments" / "AR1_phi0p5_ablate_tp",
    sample_size=sample_size,
    smoothing_grid=smoothing_grid,
    n_series=n_series,
    burn_in=burn_in,
    alpha=alpha,
    var_warmup=var_warmup_fixed,
    base_exp_kwargs=base_exp_kwargs,
    transform="student",
    rho_power=-1.0 / 3.0,
    seed_master=seed_master,
)

In [ ]:
rho_values = [-1 / 2, -1 / 3, -1 / 4, 0.0]
res_rho = run_experiment_ablation(
    base_process_template=ar1_base,
    param_name="rho_power",
    param_values=rho_values,
    outdir=Path("..") / "experiments" / "AR1_phi0p5_ablate_rho",
    sample_size=sample_size,
    smoothing_grid=smoothing_grid,
    n_series=n_series,
    burn_in=burn_in,
    alpha=alpha,
    var_warmup=var_warmup_fixed,
    base_exp_kwargs=base_exp_kwargs,
    transform="student",
    transform_power=1.0 / 3.0,
    seed_master=seed_master,
)

In [ ]:
exp_kwargs_overrides_ablation = [
    {
        "bootstrap_cls": OnlineARBootstrap,
        "method_label": "ARmmult",
        "transform": "student",
    }
]

smooth_method = "ewma"
alpha = 0.1

sample_size = 5000
smoothing_grid = [2 / 10, 2 / 20, 2 / 50, 2 / 100, 2 / 250]
n_series = 250
burn_in = 500
var_warmup = 400

base_exp_kwargs_ablation = {
    "B": 200,
    "record_every": 1,
    "smoothing_method": smooth_method,
}

phis = [0.2, 0.4, 0.6, 0.8]
panel_names_ablation = ["Stationary", "Trend + shocks"]

outdir = Path("..") / "experiments_test"
setup_dir = outdir / f"smooth-{smooth_method}_alpha-{str(alpha).replace('.', 'p')}"
abl_dir = setup_dir / "ablations_long"
abl_dir.mkdir(parents=True, exist_ok=True)

seed_master = 1234


def make_panel_process(panel_name: str, phi: float, rng: np.random.Generator):
    if panel_name == "Stationary":
        return AR1Process(
            mean=0.0,
            phi=phi,
            noise_std=1.0,
            trend_slope=0.0,
            seasonal_amplitude=0.0,
            seasonal_period=None,
            seasonal_phase=0.0,
            rng=rng,
        )
    if panel_name == "Trend + shocks":
        return AR1Process(
            mean=0.0,
            phi=phi,
            noise_std=1.0,
            trend_slope=0.001,
            seasonal_amplitude=0.0,
            seasonal_period=None,
            seasonal_phase=0.0,
            rng=rng,
            shock_type="permanent",
            jump_prob=0.005,
            jump_scale=2.0,
        )
    raise ValueError(f"Unknown panel: {panel_name}")


for panel_idx, panel_name in enumerate(panel_names_ablation):
    processes, labels = [], []
    for phi_idx, phi in enumerate(phis):
        rng = np.random.default_rng(seed_master + 1_000 * panel_idx + 10 * phi_idx)
        processes.append(make_panel_process(panel_name, phi, rng))
        labels.append(rf"phi={phi:g}")

    _ = run_sweeps_for_process_list(
        processes,
        labels=labels,
        outdir=abl_dir / panel_name,
        seed=seed_master + panel_idx,
        sample_size=sample_size,
        smoothing_grid=smoothing_grid,
        n_series=n_series,
        burn_in=burn_in,
        alpha=alpha,
        var_warmup=var_warmup,
        base_exp_kwargs=base_exp_kwargs_ablation,
        exp_kwargs_overrides=exp_kwargs_overrides_ablation,
        progress=True,
        parallel=True,
        n_jobs=-1,
    )

print("Done. Ablations saved under:", abl_dir.resolve())